# 2. Elo : calibration, lien avec les buts, stabilité

**Question** : l'espérance de résultat de l'Elo est-elle bien calibrée, et l'écart d'Elo dit-il quelque chose du nombre de buts ?

> Notebook **descriptif** (partie 3, ADR-0034). Il ne sert ni à choisir une variable ni à régler un paramètre :
> la sélection appartient au protocole de la partie 4 (validation glissante, ablation). Corrélation n'est pas apport.
> Données : période de développement seulement (matchs antérieurs au 1er juillet 2025, scellé de l'ADR-0012),
> lues par la porte `foot_predictor.features.sources`. Sorties effacées au commit (`nbstripout`) : réexécuter pour les voir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seule porte vers les données (ADR-0028) : jeu versionné et, au besoin, matchs de la période de développement.
from foot_predictor.features.sources import load_dataset, load_matches

pd.set_option("display.width", 140)
NAMES = {39: "PL", 40: "Champ.", 140: "Liga", 141: "Segunda", 78: "BL", 79: "2. BL",
         135: "Serie A", 136: "Serie B", 61: "L1", 62: "L2"}
TOP5 = [39, 140, 78, 135, 61]
frame, manifest = load_dataset()
print("Jeu de données :", manifest["version"], "-", len(frame), "lignes, jusqu'au", frame["match_day"].max())
home = frame[frame["is_home"]].copy()  # une ligne par match (celle de l'équipe à domicile)
home["total"] = home["goals_for"] + home["goals_against"]

## Calibration de l'espérance, par décile (top 5, 2005-06 à 2024-25)

E = 1 / (1 + 10^(−(écart + H)/400)), avec H figé par le réglage ; résultat observé S = 1, 0,5 ou 0.

In [ ]:
H = manifest["parameters"]["elo"]["home_advantage"]
top = home[home["api_league_id"].isin(TOP5) & (home["season_year"] >= 2005)].copy()
top["E"] = 1 / (1 + 10 ** (-(top["elo_diff"] + H) / 400))
top["S"] = np.where(top["goals_for"] > top["goals_against"], 1.0, np.where(top["goals_for"] == top["goals_against"], 0.5, 0.0))
top["decile"] = pd.qcut(top["E"], 10, labels=False)
cal = top.groupby("decile").agg(E_moyen=("E", "mean"), S_observe=("S", "mean"), matchs=("S", "size"))
display(cal.round(3))
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(cal["E_moyen"], cal["S_observe"], marker="o"); ax.plot([0.2, 0.9], [0.2, 0.9], color="grey", lw=0.8)
ax.set_xlabel("Espérance Elo (moyenne du décile)"); ax.set_ylabel("Résultat observé"); ax.set_title("Calibration par décile")
plt.show()

## Écart d'Elo et nombre de buts

In [ ]:
top["bin"] = pd.cut(top["elo_diff"], [-600, -200, -100, -50, 0, 50, 100, 200, 600])
link = top.groupby("bin", observed=True).agg(total=("total", "mean"), buts_dom=("goals_for", "mean"),
                                             buts_ext=("goals_against", "mean"), matchs=("total", "size"))
display(link.round(3))
top["abs_bin"] = pd.cut(top["elo_diff"].abs(), [0, 50, 100, 200, 600], include_lowest=True)
display(top.groupby("abs_bin", observed=True)["total"].agg(["mean", "size"]).round(3))

## Stabilité d'une saison à l'autre

Corrélation entre la note moyenne d'une équipe dans une saison et dans la suivante (équipes présentes les deux saisons, même championnat ou non).

In [ ]:
season_mean = frame.groupby(["team_id", "season_year"])["elo_pre"].mean().reset_index()
nxt = season_mean.assign(season_year=season_mean["season_year"] - 1)
pair = season_mean.merge(nxt, on=["team_id", "season_year"], suffixes=("", "_suivante"))
print("paires :", len(pair), "- corrélation :", round(pair["elo_pre"].corr(pair["elo_pre_suivante"]), 3))
change = (pair["elo_pre_suivante"] - pair["elo_pre"])
display(change.describe().round(1))

## Conclusion

- **Calibration** : l'espérance brute de l'Elo, avec l'avantage du terrain réglé (40 points), est bien *ordonnée* (le résultat observé croît à chaque décile) mais trop *extrême* : 0,19 contre 0,28 observé dans le décile le plus bas, 0,87 contre 0,85 dans le plus haut. Ce n'est pas un défaut du réglage : celui-ci passait par une logistique ordonnée qui recalibre l'écart. En partie 4, l'écart d'Elo entre comme variable d'un modèle, pas comme probabilité.
- **Lien avec les buts** : le total croît avec l'écart *absolu* (2,54 buts quand |écart| ≤ 50 points, 3,03 au-delà de 200). Un match déséquilibré donne plus de buts, surtout au favori : 2,47 buts à domicile quand l'équipe à domicile est favorite de plus de 200 points. Le lien n'est donc pas linéaire en l'écart signé pour le total, alors qu'il l'est à peu près pour les buts de chaque équipe.
- **Stabilité** : corrélation de 0,957 entre la note moyenne d'une équipe dans une saison et dans la suivante. Variation médiane nulle, écart type 40 points : c'est cohérent avec r = 0 (aucun retour à la moyenne à l'intersaison).